# 07 — Three-Fold Cross-Validation for the 04.5 Model

## Objective

A single 70/30 holdout can be unusually easy or difficult. This notebook estimates the stability of the 04.5 LightGBM model with stratified three-fold cross-validation.

The dataset already contains one row per customer, so every customer appears in exactly one validation fold. Concatenating the three validation predictions produces out-of-fold (OOF) probabilities for the full training population.


In [ ]:
from pathlib import Path

def find_project_root() -> Path:
    """Use the parent of the notebook working directory as the project root."""
    project_root = Path.cwd().resolve().parent
    if not (project_root / 'data').is_dir():
        raise FileNotFoundError(
            "Expected the project data directory one level above the current working directory."
        )
    return project_root

import gc, json, platform, time
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
print('Python:', platform.python_version())
print('LightGBM:', lgb.__version__)


## 1. Load the reference customer table

The notebook confirms one-row-per-customer granularity before any split. IDs and targets are retained for alignment; all remaining columns form the 1,022-feature matrix.


In [ ]:
ROOT = find_project_root()
STEP_DIR = ROOT / '07_baseline_cv'
DATA_PATH = ROOT / '04_5_temporal_refined' / 'train_aggregated_temporal_refined.feather'
HOLDOUT_METRICS_PATH = ROOT / '04_5_temporal_refined' / 'temporal_refined_metrics.json'
FOLD_METRICS_PATH = STEP_DIR / 'lightgbm_04_5_cv_fold_metrics.csv'
CV_METRICS_PATH = STEP_DIR / 'lightgbm_04_5_cv_metrics.json'
OOF_PATH = STEP_DIR / 'lightgbm_04_5_oof_predictions.parquet'
IMPORTANCE_PATH = STEP_DIR / 'lightgbm_04_5_cv_feature_importance.csv'
for path in [DATA_PATH, HOLDOUT_METRICS_PATH]:
    assert path.exists(), f'找不到：{path}'

data = pd.read_feather(DATA_PATH)
assert data['customer_ID'].is_unique
assert len(data) == data['customer_ID'].nunique()
assert data.shape[1] - 2 == 1022
print(f'Data: {len(data):,} rows = {data.customer_ID.nunique():,} unique customers')
print('Features:', data.shape[1] - 2)
display(data['target'].value_counts().sort_index().to_frame('customers'))


## 2. Define the metric and unchanged model

`StratifiedKFold(n_splits=3, shuffle=True, random_state=42)` preserves the default rate while assigning each customer to one validation fold.

The model uses the original 04.5 parameters. This stage estimates the reference configuration rather than introducing another experimental change.


In [ ]:
customer_ids = data['customer_ID'].copy()
y = data['target'].astype('int8').copy()
X = data.drop(columns=['customer_ID','target']).copy()
del data; gc.collect()
X['S_2_last'] = (pd.to_datetime(X['S_2_last']) - pd.Timestamp('2018-03-01')).dt.days.astype('int16')
CATEGORICAL_VARIABLES = ['B_30','B_38','D_114','D_116','D_117','D_120','D_126','D_63','D_64','D_66','D_68']
CATEGORICAL_FEATURES = [f'{c}_last' for c in CATEGORICAL_VARIABLES]
for column in CATEGORICAL_FEATURES:
    X[column] = X[column].astype('category')

def amex_metric(y_true, y_pred):
    y_true=np.asarray(y_true,dtype=np.int8); y_pred=np.asarray(y_pred,dtype=np.float64)
    weights=np.where(y_true==0,20.0,1.0)
    order=np.argsort(-y_pred,kind='mergesort'); yy,ww=y_true[order],weights[order]
    top=np.cumsum(ww)<=.04*ww.sum()
    top_four=(yy[top]*ww[top]).sum()/(y_true*weights).sum()
    def gini(pred):
        idx=np.argsort(-pred,kind='mergesort'); sy,sw=y_true[idx],weights[idx]
        return np.sum((np.cumsum(sy*sw)/np.sum(sy*sw)-np.cumsum(sw)/np.sum(sw))*sw)
    return .5*(gini(y_pred)/gini(y_true)+top_four)

def make_model(seed):
    return lgb.LGBMClassifier(objective='binary',n_estimators=1500,learning_rate=.03,num_leaves=31,max_depth=-1,min_child_samples=100,subsample=.8,colsample_bytree=.8,reg_lambda=1.0,max_bin=127,random_state=seed,n_jobs=4,verbosity=-1)

cv = StratifiedKFold(n_splits=3,shuffle=True,random_state=42)
print('Model parameters:', make_model(42).get_params())


## 3. Train each fold

For every fold, the code verifies zero customer overlap, trains with early stopping, saves the best booster, predicts the held-out customers, and records metrics and feature importance.

OOF arrays are populated only at validation indices, ensuring that no customer's prediction comes from a model trained on that customer.


In [ ]:
oof_probability = np.full(len(X), np.nan, dtype='float32')
oof_fold = np.zeros(len(X), dtype='int8')
fold_results, importance_parts = [], []
total_started = time.perf_counter()

for fold, (train_idx, valid_idx) in enumerate(cv.split(X,y), start=1):
    fold_started = time.perf_counter()
    train_ids = customer_ids.iloc[train_idx]
    valid_ids = customer_ids.iloc[valid_idx]
    overlap = len(set(train_ids).intersection(set(valid_ids)))
    assert overlap == 0
    assert len(train_ids) + len(valid_ids) == len(X)
    print(f'\nFold {fold}/3 — train {len(train_idx):,}, valid {len(valid_idx):,}, overlap {overlap}')

    model = make_model(42)
    model.fit(X.iloc[train_idx],y.iloc[train_idx],eval_set=[(X.iloc[valid_idx],y.iloc[valid_idx])],eval_metric='auc',categorical_feature=CATEGORICAL_FEATURES,callbacks=[lgb.early_stopping(100,verbose=True),lgb.log_evaluation(100)])
    probability = model.predict_proba(X.iloc[valid_idx],num_iteration=model.best_iteration_)[:,1]
    oof_probability[valid_idx] = probability.astype('float32')
    oof_fold[valid_idx] = fold
    elapsed = time.perf_counter()-fold_started
    result={'fold':fold,'train_customers':len(train_idx),'valid_customers':len(valid_idx),'customer_overlap':overlap,'positive_rate':float(y.iloc[valid_idx].mean()),'roc_auc':float(roc_auc_score(y.iloc[valid_idx],probability)),'amex_metric':float(amex_metric(y.iloc[valid_idx],probability)),'best_iteration':int(model.best_iteration_),'training_seconds':float(elapsed)}
    fold_results.append(result)
    fold_table=pd.DataFrame({'feature':model.booster_.feature_name(),f'fold_{fold}_gain':model.booster_.feature_importance(importance_type='gain'),f'fold_{fold}_split':model.booster_.feature_importance(importance_type='split')})
    importance_parts.append(fold_table)
    model.booster_.save_model(STEP_DIR/f'lightgbm_04_5_cv_fold_{fold}.txt',num_iteration=model.best_iteration_)
    pd.DataFrame(fold_results).to_csv(FOLD_METRICS_PATH,index=False)
    print(result)
    del model, probability, train_ids, valid_ids; gc.collect()

total_seconds=time.perf_counter()-total_started
assert not np.isnan(oof_probability).any()
assert set(oof_fold)=={1,2,3}
fold_metrics=pd.DataFrame(fold_results)
display(fold_metrics)


## 4. Summarize stability and OOF performance

Fold means and standard deviations quantify sensitivity to sampling. The concatenated OOF predictions provide the most useful single estimate because they cover every customer under a consistent out-of-sample protocol.

Variation in the AMEX metric is larger than variation in ROC-AUC, which is expected because top-4% capture depends strongly on a relatively small high-risk slice.


In [ ]:
summary={
 'fold_roc_auc_mean':float(fold_metrics.roc_auc.mean()),'fold_roc_auc_std':float(fold_metrics.roc_auc.std(ddof=1)),
 'fold_amex_mean':float(fold_metrics.amex_metric.mean()),'fold_amex_std':float(fold_metrics.amex_metric.std(ddof=1)),
 'oof_roc_auc':float(roc_auc_score(y,oof_probability)),'oof_amex_metric':float(amex_metric(y,oof_probability)),
 'best_iteration_mean':float(fold_metrics.best_iteration.mean()),'best_iteration_std':float(fold_metrics.best_iteration.std(ddof=1)),
 'total_training_seconds':float(total_seconds),'customer_count':len(X),'feature_count':X.shape[1]
}
holdout=json.loads(HOLDOUT_METRICS_PATH.read_text())['temporal_refined']
comparison=pd.DataFrame([holdout,summary],index=['04.5 single holdout','04.5 3-fold CV'])
display(comparison); print(summary)

importance=importance_parts[0]
for part in importance_parts[1:]: importance=importance.merge(part,on='feature',validate='one_to_one')
gain_cols=[f'fold_{i}_gain' for i in range(1,4)]; split_cols=[f'fold_{i}_split' for i in range(1,4)]
importance['mean_gain']=importance[gain_cols].mean(axis=1); importance['std_gain']=importance[gain_cols].std(axis=1,ddof=1)
importance['mean_split']=importance[split_cols].mean(axis=1); importance['gain_share']=importance.mean_gain/importance.mean_gain.sum()
importance=importance.sort_values(['mean_gain','feature'],ascending=[False,True]).reset_index(drop=True); importance['rank']=np.arange(1,len(importance)+1)
importance.to_csv(IMPORTANCE_PATH,index=False)
display(importance.head(30))


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Fold stability around the OOF benchmark
cv_plot = fold_metrics.melt(id_vars='fold', value_vars=['roc_auc', 'amex_metric'], var_name='metric', value_name='score')
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
for ax, metric, label in zip(axes, ['roc_auc', 'amex_metric'], ['ROC-AUC', 'AMEX metric']):
    subset = cv_plot[cv_plot['metric'] == metric]
    sns.pointplot(data=subset, x='fold', y='score', color='#4C78A8', markers='o', linestyles='-', ax=ax)
    oof_value = summary['oof_roc_auc'] if metric == 'roc_auc' else summary['oof_amex_metric']
    ax.axhline(oof_value, color='#E45756', linestyle='--', label=f'OOF = {oof_value:.6f}')
    ax.set(title=f'{label} across validation folds', xlabel='Fold', ylabel=label)
    ax.legend(frameon=False)
sns.despine()
plt.tight_layout()
plt.show()


## 5. Save the cross-validation benchmark

The notebook saves fold models, fold metrics, averaged gain importance, full OOF predictions, and a structured summary. The fold assignment is reused later so tuned candidates are tested on exactly the same customers.


In [ ]:
pd.DataFrame({'customer_ID':customer_ids.to_numpy(),'target':y.to_numpy(),'fold':oof_fold,'oof_probability':oof_probability}).to_parquet(OOF_PATH,index=False,compression='zstd')
output={'folds':fold_results,'summary':summary,'single_holdout_04_5':holdout,'split':{'method':'StratifiedKFold','n_splits':3,'shuffle':True,'random_state':42,'customer_level':True}}
CV_METRICS_PATH.write_text(json.dumps(output,indent=2),encoding='utf-8')
for path in [FOLD_METRICS_PATH,CV_METRICS_PATH,OOF_PATH,IMPORTANCE_PATH,*[STEP_DIR/f'lightgbm_04_5_cv_fold_{i}.txt' for i in range(1,4)]]: print(path.name,'->',path.resolve())
